## Process the customers data
###### 1. Ingest the data into the data lakehouse - bronze_customers
###### 2. Perform data quality checks and transform the data as required - siver_customers_clean
###### 3. apply changes to the Customers data - silver_customers

### 1. Ingest the data into the data lakehouse - bronze_customers

###### Add columns
###### 1. Input_file_path
###### 2. Ingestion_timestamp

In [0]:
create or refresh streaming table bronze_customers
comment 'raw customers data ingested from the source system operations data'
tblproperties('quality' = 'bronze')
As 
select * , 
        _metadata.file_path as input_file_path,
        CURRENT_TIMESTAMP as ingestion_timestamp
from cloud_files(
  '/Volumes/circuitbox/landing/operations_data/customers',
  'json',
  map('cloudFiles.inferColumnTypes', 'true')
);




#### 2. Perform data quality chekcks and transform the data as required - silver_customers_clean

Siver Customer clean (DQ Rules) [Data Quality Rules]
1. Fail if customer_id is NULL
2. Drop records with customer_name as Null
3. Warn if telephone is less than 10 characters
4. Warn if email is NULL
5. Warn if date_of_birth is befor 1920

Silver Customer Clean (Transofrmations)
1. CAST date_of_birth to DATE
2. CAST created_date to DATE


######## Source --> For each Reocrd --> Expectation --> true --> Inclued Repecord --> Target
########                            --> Expectation --> False--> Action --> Warn --> Inclued Repecord --> Target
########                                                     --> Action --> Fail --> Fail Data Flow
########                                                     --> Action --> Drop --> Drop Record

CONSTRAINT <expectation_name>
    EXPECT (Conditonal_expresssion>)
    [on Violation (Fail UPDATE /\ Drop Row)]    


In [0]:
Create or refresh streaming table silver_customers_clean(
CONSTRAINT valid_cistomer_id expect (customer_id is not null) on violation fail update,
CONSTRAINT valid_customer_name expect (customer_name is not null) on violation drop row,
CONSTRAINT valid_telephone expect (length(telephone)>=10 is not null) ,
constraint valid_email expect (email is not null),
constraint valid_date_of_birth expect (date_of_birth >= '1920-01-01') 
)
comment 'Cleaned customer data'
tblproperties('quality' = 'silver')
as
select customer_id, 
customer_name, 
CAST(date_of_birth AS DATE) as date_of_birth, 
email, 
telephone, 
CAST(created_date AS DATE) as created_date
from STREAM(LIVE.bronze_customers)



LIVE. is only applicable for table with the same delta pipelines

Stream - only gets the incremental data

###### 3. apply changes to the Customers data - silver_customers

SCD - Slowly Changing dimensions

Cutomer --> Order --> Product

Customer --> Who - Customer Dimension
Product --> what --> Product Dimenstion

Order --> Fact table

SCD Type 1 - Overwrite the data (No History Maintained)
SCD Type 2 - Full History Maintained  ... uses end_date and new record will be created.
SCD Type 3 - only the most recent change is tracked --- new and old columns are maintainted in the table




Silver Customers is Type 1 Dimension Table. with Primary key as customer_id. Apply changes to the customer data based o Created_date.

Apply Changes API is the concept used 

CREATE OR REFRSH STREAMING ABLE <target_table>;

APPLY Changes into <target_type>

From <source_table>

Keys <columns>

Sequence BY <colunmns>

Stored as <SCD Type 1 | SDC Type 2>

Apply Changed into Live.table_name

From Source

keys(keys)

[IGNORE Null Update]

[Applu as Delete when conditon]

[Apply AS Truncate when condition]

Sequence By OrderByColumn

[Columns {columnList |*Except (exceptcolumnList)}]

[Stored as {SCD Type 1 | SCD Type 2}]

[Track History on {columnList | *EXcept (except columnList)}]

In [0]:
CREATE Or Refresh STREAMING TABLE silver_customers
COMMENT 'SCD Type 1 customer data'
TBLPROPERTIES('quality' = 'silver')

In [0]:
APPLY CHANGES INTO LIVE.silver_customers
from stream(Live.silver_customers_clean)
keys (customer_id)
sequence by created_date
stored as scd type 1; --optional. type 1 is the default value
